# Real U50 A-only Benchmark: GAP + MILP + Genetic

Бенчмарк в стиле `real_full_3algo_benchmark.ipynb` для упрощенного датасета:
`src/data/real/day_load_profiles/u50/simplified_a_only/dataset_real_spb_day_u50_A_only_simplified.json`.

Выполняет 3 алгоритма (`GAP-VRP`, `REAL-MILP`, `REAL-GENETIC`), собирает общие метрики,
детальные параметры запуска и breakdown `агент -> задачи`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import importlib
import inspect
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Принудительно вычищаем старые flowopt-модули из kernel-кеша
for mod_name in list(sys.modules):
    if mod_name.startswith("flowopt"):
        sys.modules.pop(mod_name, None)

import flowopt.pipeline as fp
fp = importlib.reload(fp)

DATA_ROOT = fp.DATA_ROOT
run_real_gap_vrp = fp.run_real_gap_vrp
run_real_milp = fp.run_real_milp
run_real_genetic = fp.run_real_genetic

milp_sig = inspect.signature(run_real_milp)
if "time_limit_sec" not in milp_sig.parameters:
    raise RuntimeError(f"Stale run_real_milp loaded: signature={milp_sig}")

pd.set_option("display.max_colwidth", 160)
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("pipeline module:", fp.__file__)
print("run_real_milp signature:", milp_sig)


In [ ]:
import json

NOTEBOOK_DIR = REPO_ROOT / "notebooks"
DATASET_PATH = REPO_ROOT / "src" / "data" / "real" / "day_load_profiles" / "u50" / "simplified_a_only" / "dataset_real_spb_day_u50_A_only_simplified.json"

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"DATASET_PATH not found: {DATASET_PATH}")

print("DATASET_PATH:", DATASET_PATH)

with DATASET_PATH.open("r", encoding="utf-8") as f:
    dataset = json.load(f)

meta = dataset.get("metadata", {})
counts = {
    "nodes": len(dataset.get("graph", {}).get("nodes", [])),
    "edges": len(dataset.get("graph", {}).get("edges", [])),
    "agents": len(dataset.get("agents", [])),
    "tasks": len(dataset.get("tasks", [])),
    "objects": len({t.get("destination_node_id") for t in dataset.get("tasks", [])}),
    "sources": len({t.get("source_node_id") for t in dataset.get("tasks", [])}),
}

print("dataset name:", meta.get("name"))
print("counts:", counts)
print("simplified_profile:", meta.get("simplified_profile", {}))

pd.DataFrame([counts])


# Optional: align solver limits/speeds with dataset vehicle_profiles
APPLY_PROFILE_LIMITS = True
if APPLY_PROFILE_LIMITS:
    import flowopt.core as core
    import flowopt.genetic_solver_components as ga

    profiles = meta.get("vehicle_profiles", {})
    for vt, p in profiles.items():
        km = float(p.get("max_daily_km", core.MAX_DAILY_KM_BY_TYPE.get(vt, 130.0)))
        hh = float(p.get("max_shift_hours", core.MAX_SHIFT_HOURS_BY_TYPE.get(vt, 10.0)))
        sp = float(p.get("avg_speed_kmph", core.AVG_SPEED_KMPH_BY_TYPE.get(vt, 24.0)))
        core.MAX_DAILY_KM_BY_TYPE[vt] = km
        core.MAX_SHIFT_HOURS_BY_TYPE[vt] = hh
        core.AVG_SPEED_KMPH_BY_TYPE[vt] = sp
        ga.MAX_DAILY_KM_BY_TYPE[vt] = km
        ga.MAX_SHIFT_HOURS_BY_TYPE[vt] = hh
        ga.AVG_SPEED_KMPH_BY_TYPE[vt] = sp

    print("Patched limits from vehicle_profiles for:", sorted(profiles.keys()))
    print("KM limits:", {k: core.MAX_DAILY_KM_BY_TYPE.get(k) for k in ["VT_A", "VT_AB", "VT_ABD", "VT_AD"]})
    print("Hours limits:", {k: core.MAX_SHIFT_HOURS_BY_TYPE.get(k) for k in ["VT_A", "VT_AB", "VT_ABD", "VT_AD"]})


In [ ]:
# GAP
GAP_STEP1_METHOD = "lp"
GAP_ITER = 25
MAX_AGENTS = None
SHOW_PROGRESS = False

# Real MILP
RMILP_TIME_LIMIT_SEC = 120
RMILP_UNASSIGNED_PENALTY = 1e7

# Genetic
GA_POPULATION_SIZE = 80
GA_GENERATIONS = 120
GA_ELITE_SIZE = 6
GA_SEED = 42
GA_GENERATION_SCALE = 1.0
GA_MIN_GENERATIONS = 20

# Live progress in notebook output
SHOW_ALGO_PROGRESS = True
SHOW_SOLVER_DETAILS = False

from time import perf_counter


def make_progress_logger(enabled: bool):
    if not enabled:
        return None
    t0 = perf_counter()
    def _log(message: str) -> None:
        dt = perf_counter() - t0
        print(f"[+{dt:7.1f}s] {message}", flush=True)
    return _log


progress_log = make_progress_logger(SHOW_ALGO_PROGRESS)


def run_with_live_status(label, fn, **kwargs):
    if progress_log is not None:
        progress_log(f"{label}: START")
    ts = perf_counter()
    out = fn(**kwargs)
    if progress_log is not None:
        progress_log(f"{label}: DONE in {perf_counter() - ts:.1f}s")
    return out


results = [
    run_with_live_status(
        "GAP-VRP",
        run_real_gap_vrp,
        dataset_path=DATASET_PATH,
        step1_method=GAP_STEP1_METHOD,
        gap_iter=GAP_ITER,
        max_agents=MAX_AGENTS,
        use_repair=True,
        show_progress=SHOW_SOLVER_DETAILS,
        verbose=False,
        progress_hook=progress_log,
    ),
    run_with_live_status(
        "REAL-MILP",
        run_real_milp,
        dataset_path=DATASET_PATH,
        time_limit_sec=RMILP_TIME_LIMIT_SEC,
        unassigned_penalty=RMILP_UNASSIGNED_PENALTY,
        show_progress=SHOW_SOLVER_DETAILS,
        progress_hook=progress_log,
    ),
    run_with_live_status(
        "REAL-GENETIC",
        run_real_genetic,
        dataset_path=DATASET_PATH,
        population_size=GA_POPULATION_SIZE,
        generations=GA_GENERATIONS,
        elite_size=GA_ELITE_SIZE,
        seed=GA_SEED,
        generation_scale=GA_GENERATION_SCALE,
        min_generations=GA_MIN_GENERATIONS,
        show_progress=SHOW_SOLVER_DETAILS,
        progress_hook=progress_log,
    ),
]

benchmark_df = pd.DataFrame([r.as_dict() for r in results])
benchmark_df = benchmark_df.sort_values(
    by=["all_checks_ok", "transport_work_ton_km", "total_km", "runtime_sec"],
    ascending=[False, True, True, True],
    na_position="last",
).reset_index(drop=True)

# Save benchmark artifact to local JSON
from datetime import datetime

LOCAL_OUT_DIR = REPO_ROOT / "notebooks" / "local" / "real_data" / "u50_aonly"
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULT_JSON_PATH = LOCAL_OUT_DIR / f"benchmark_u50_aonly_{RUN_TAG}.json"

records = benchmark_df.where(pd.notna(benchmark_df), None).to_dict(orient="records")
artifact = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebook": "real_u50_aonly_3algo_benchmark.ipynb",
    "dataset_path": str(DATASET_PATH),
    "config": {
        "gap_step1_method": GAP_STEP1_METHOD,
        "gap_iter": GAP_ITER,
        "max_agents": MAX_AGENTS,
        "rmilp_time_limit_sec": RMILP_TIME_LIMIT_SEC,
        "rmilp_unassigned_penalty": RMILP_UNASSIGNED_PENALTY,
        "ga_population_size": GA_POPULATION_SIZE,
        "ga_generations": GA_GENERATIONS,
        "ga_elite_size": GA_ELITE_SIZE,
        "ga_seed": GA_SEED,
        "ga_generation_scale": GA_GENERATION_SCALE,
        "ga_min_generations": GA_MIN_GENERATIONS,
        "show_algo_progress": SHOW_ALGO_PROGRESS,
        "show_solver_details": SHOW_SOLVER_DETAILS,
    },
    "results": records,
}
RESULT_JSON_PATH.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", RESULT_JSON_PATH)

main_cols = [
    "algorithm",
    "feasible",
    "all_checks_ok",
    "assigned_routes",
    "unassigned_tasks",
    "active_agents",
    "transport_work_ton_km",
    "total_km",
    "deadhead_km",
    "deadhead_share_pct",
    "total_hours",
    "runtime_sec",
    "solver_error",
]
benchmark_df[[c for c in main_cols if c in benchmark_df.columns]]


In [ ]:
# Детали проверок и параметров по каждому алгоритму
detail_cols = [
    "algorithm",
    "checks",
    "solver_error",
    "step1_method",
    "gap_iter",
    "used_agents",
    "time_limit_sec",
    "unassigned_penalty",
    "population_size",
    "generations",
    "generations_requested",
    "generation_scale",
    "elite_size",
]
benchmark_df[[c for c in detail_cols if c in benchmark_df.columns]]


In [ ]:
from flowopt.reporting import solution_breakdown_tables
from IPython.display import Markdown, display

MAX_AGENTS_TO_SHOW = 30
MAX_TASK_IDS_IN_CELL = 12
MAX_TASK_ROWS_TO_SHOW = 400

for _, row in benchmark_df.iterrows():
    algo = row.get("algorithm", "unknown")
    tables = solution_breakdown_tables(
        row,
        max_agents=MAX_AGENTS_TO_SHOW,
        max_task_ids=MAX_TASK_IDS_IN_CELL,
    )

    display(Markdown(f"### {algo}: решение по агентам"))
    display(tables["summary"])

    if tables["agents"].empty:
        print("No active agents in solution")
        continue

    display(tables["agents"])
    display(Markdown(f"**{algo}: задача -> агент (первые {MAX_TASK_ROWS_TO_SHOW} строк)**"))
    display(tables["tasks"].head(MAX_TASK_ROWS_TO_SHOW))
